In [ ]:

import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

data = {
    'text': ["I loved this movie!", "Worst experience ever.", "It was okay.", "Fantastic acting!", "Never again."],
    'label': [1, 0, 1, 1, 0] 
}
df = pd.DataFrame(data)

train_texts, temp_texts, train_labels, temp_labels = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)
val_texts, test_texts, val_labels, test_labels = train_test_split(temp_texts, temp_labels, test_size=0.5, random_state=42)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128, return_tensors="pt")

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)
test_encodings = tokenize_function(test_texts)

class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = SimpleDataset(train_encodings, train_labels)
val_dataset = SimpleDataset(val_encodings, val_labels)
test_dataset = SimpleDataset(test_encodings, test_labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

for name, param in model.bert.named_parameters():
    if not any(nd in name for nd in ["layer.10", "layer.11"]):
        param.requires_grad = False

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    logging_dir='./logs',
    evaluation_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results = trainer.evaluate(test_dataset)
print("\n--- Test Set Evaluation ---")
for key, value in results.items():
    print(f"{key}: {value}")

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)
print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, preds))